#1: Importing Libraries and Loading Data

In [1]:
import pandas as pd

# Load the datasets
customers_df = pd.read_csv("Day9_Customers.csv")
orders_df = pd.read_csv("Day9_Orders.csv")
products_df = pd.read_csv("Day9_Products.csv")

# Inspect the loaded data
print("--- Customers Dataset ---")
display(customers_df.head(2))

print("\n--- Orders Dataset ---")
display(orders_df.head(2))

print("\n--- Products Dataset ---")
display(products_df.head(2))

--- Customers Dataset ---


,Customer_ID,Customer_Name,City,Region,Membership_Type
0,C001,Aarav Sharma,Srinagar,North,Premium
1,C002,Zoya Khan,Delhi,North,Regular



--- Orders Dataset ---


,Order_ID,Order_Date,Customer_ID,Product_ID,Quantity,Payment_Method,Order_Status
0,O0001,2026-02-19,C027,P019,2,Credit Card,Delivered
1,O0002,2026-01-25,C006,P003,2,Debit Card,Delivered



--- Products Dataset ---


,Product_ID,Product_Name,Category,Unit_Price,Brand
0,P001,Wireless Headphones,Electronics,1499,SoundMax
1,P002,Mechanical Keyboard,Electronics,2499,KeyPro


#2: Demonstrating concat()

In [2]:
# Split the orders into two halves to demonstrate concatenation
orders_part1 = orders_df.iloc[:len(orders_df)//2]
orders_part2 = orders_df.iloc[len(orders_df)//2:]

print(f"Part 1 shape: {orders_part1.shape}")
print(f"Part 2 shape: {orders_part2.shape}")

# Recombine using pd.concat()
combined_orders = pd.concat([orders_part1, orders_part2], ignore_index=True)
print(f"Recombined Orders shape: {combined_orders.shape}")

Part 1 shape: (60, 7)
Part 2 shape: (60, 7)
Recombined Orders shape: (120, 7)


#3: Merging the DataFrames

In [3]:
# Merge Orders and Customers on Customer_ID
merged_df = pd.merge(combined_orders, customers_df, on='Customer_ID', how='left')

# Merge the result with Products on Product_ID
final_merged_df = pd.merge(merged_df, products_df, on='Product_ID', how='left')

print("--- Data After Merging ---")
display(final_merged_df.head())

--- Data After Merging ---


,Order_ID,Order_Date,Customer_ID,Product_ID,Quantity,Payment_Method,Order_Status,Customer_Name,City,Region,Membership_Type,Product_Name,Category,Unit_Price,Brand
0,O0001,2026-02-19,C027,P019,2,Credit Card,Delivered,Harsh Vardhan,Noida,North,Premium,Cricket Bat,Sports,2499,BatPro
1,O0002,2026-01-25,C006,P003,2,Debit Card,Delivered,Ishita Gupta,Bengaluru,South,Premium,Wireless Mouse,Electronics,899,TechGear
2,O0003,2026-02-26,C015,P004,1,Cash on Delivery,Delivered,Karan Joshi,Chandigarh,North,Regular,Smart Watch,Electronics,3299,FitTech
3,O0004,2026-03-04,C024,P015,3,Net Banking,Delivered,Maryam Khan,Hyderabad,South,Regular,Machine Learning Basics,Books,999,AIPress
4,O0005,2026-03-29,C025,P009,5,Credit Card,Delivered,Reyansh Jain,Kolkata,East,New,Coffee Maker,Home & Kitchen,3499,HomeBrew


#4: Applying DateTime Operations

In [4]:
# Convert 'Order_Date' to a datetime object
final_merged_df['Order_Date'] = pd.to_datetime(final_merged_df['Order_Date'])

# Extract Month, Day, and Day of the Week
final_merged_df['Order_Month'] = final_merged_df['Order_Date'].dt.month_name()
final_merged_df['Order_Day'] = final_merged_df['Order_Date'].dt.day
final_merged_df['Order_Day_of_Week'] = final_merged_df['Order_Date'].dt.day_name()

print("--- Data After DateTime Extraction ---")
display(final_merged_df[['Order_Date', 'Order_Month', 'Order_Day', 'Order_Day_of_Week']].head())

--- Data After DateTime Extraction ---


,Order_Date,Order_Month,Order_Day,Order_Day_of_Week
0,2026-02-19,February,19,Thursday
1,2026-01-25,January,25,Sunday
2,2026-02-26,February,26,Thursday
3,2026-03-04,March,4,Wednesday
4,2026-03-29,March,29,Sunday


#5: Using apply() to Create Transformed Columns

In [5]:
# Calculate Total Amount using apply() with lambda
final_merged_df['Total_Amount'] = final_merged_df.apply(
    lambda row: row['Quantity'] * row['Unit_Price'],
    axis=1
)

# Apply a 10% discount for Premium members using apply()
def calculate_discount(row):
    if row['Membership_Type'] == 'Premium':
        return row['Total_Amount'] * 0.90 # 10% off
    return row['Total_Amount']

final_merged_df['Final_Payable_Amount'] = final_merged_df.apply(calculate_discount, axis=1)

print("--- Data After apply() Transformations ---")
display(final_merged_df[['Customer_Name', 'Membership_Type', 'Quantity', 'Unit_Price', 'Total_Amount', 'Final_Payable_Amount']].head())

--- Data After apply() Transformations ---


,Customer_Name,Membership_Type,Quantity,Unit_Price,Total_Amount,Final_Payable_Amount
0,Harsh Vardhan,Premium,2,2499,4998,4498.2
1,Ishita Gupta,Premium,2,899,1798,1618.2
2,Karan Joshi,Regular,1,3299,3299,3299.0
3,Maryam Khan,Regular,3,999,2997,2997.0
4,Reyansh Jain,New,5,3499,17495,17495.0


#6: Cleaning up and Exporting the Processed Dataset

In [6]:
# Reorder columns for a clean, meaningful structure
columns_order = [
    'Order_ID', 'Order_Date', 'Order_Month', 'Order_Day_of_Week',
    'Customer_Name', 'City', 'Membership_Type',
    'Product_Name', 'Category', 'Brand', 'Quantity', 'Unit_Price',
    'Total_Amount', 'Final_Payable_Amount', 'Payment_Method', 'Order_Status'
]

# Create the final processed DataFrame
processed_dataset = final_merged_df[columns_order]

print("--- Final Processed Dataset ---")
display(processed_dataset.head())

# Export the processed dataset to a new CSV file
processed_dataset.to_csv('Day9_Processed_ECommerce_Dataset.csv', index=False)
print("\nDataset successfully processed and exported as 'Day9_Processed_ECommerce_Dataset.csv'!")

--- Final Processed Dataset ---


,Order_ID,Order_Date,Order_Month,Order_Day_of_Week,Customer_Name,City,Membership_Type,Product_Name,Category,Brand,Quantity,Unit_Price,Total_Amount,Final_Payable_Amount,Payment_Method,Order_Status
0,O0001,2026-02-19,February,Thursday,Harsh Vardhan,Noida,Premium,Cricket Bat,Sports,BatPro,2,2499,4998,4498.2,Credit Card,Delivered
1,O0002,2026-01-25,January,Sunday,Ishita Gupta,Bengaluru,Premium,Wireless Mouse,Electronics,TechGear,2,899,1798,1618.2,Debit Card,Delivered
2,O0003,2026-02-26,February,Thursday,Karan Joshi,Chandigarh,Regular,Smart Watch,Electronics,FitTech,1,3299,3299,3299.0,Cash on Delivery,Delivered
3,O0004,2026-03-04,March,Wednesday,Maryam Khan,Hyderabad,Regular,Machine Learning Basics,Books,AIPress,3,999,2997,2997.0,Net Banking,Delivered
4,O0005,2026-03-29,March,Sunday,Reyansh Jain,Kolkata,New,Coffee Maker,Home & Kitchen,HomeBrew,5,3499,17495,17495.0,Credit Card,Delivered



Dataset successfully processed and exported as 'Day9_Processed_ECommerce_Dataset.csv'!
